In [28]:
# Imports for environment variables
from dotenv import load_dotenv
import os

# Imports for Data Ingestion (Source, Load)
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Imports for chunking (Transform)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Imports for embeddings (Embed)
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

# Imports for storing the data (Vector DB)
from langchain_community.vectorstores import Chroma

# Imports for retrieving the data
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import RetrievalQA


In [9]:
# PDF Loader
loader = PyPDFDirectoryLoader('./us-census')
docs = loader.load()

CHUNK_SIZE = 500
CHUNK_OVERLAP = 120

text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
final_documents = text_splitter.split_documents(docs)

len(final_documents)

646

In [10]:
# Embeddings using hugging face
huggingface_embeddings = HuggingFaceBgeEmbeddings(
  model_name='sentence-transformers/all-MiniLM-L6-v2',
  model_kwargs={'device': 'cpu'},
  encode_kwargs={'normalize_embeddings': True}
)

C:\Users\user\AppData\Local\Temp\ipykernel_12628\267882098.py:2: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings = HuggingFaceBgeEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 734.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
# Vector DB
chroma_vectordb = Chroma.from_documents(final_documents, huggingface_embeddings)

In [22]:
# Doing similarity search
result = chroma_vectordb.similarity_search(query='where is this data coming from?')
result[2].page_content

'of categorization. For more information \non data classification methods, refer to \n<https:/ /pro.arcgis.com/en/pro-app/latest/\nhelp/mapping/layer-properties/data-\nclassification-methods.htm>.\nFigure /one.tab/period.tab\nAmerican Community Survey Poverty Rates/colon.tab /two.tab/zero.tab/zero.tab/five.tab to /two.tab/zero.tab/two.tab/two.tab\n(In percent)\nNote: Estimates for 2020 experimental data are unavailable. For more information, refer to'

In [32]:
# Retriever

# Load api key
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

retriever = chroma_vectordb.as_retriever(
  search_type="similarity",
  search_kwargs={"k": 5}
)

# LLM
llm = ChatGroq(model_name='llama-3.3-70b-versatile')
prompt = PromptTemplate.from_template(
  """
    Answer the questions based only on the following context. Don't hallucinate. Always give accurate answers. Think before giving the answer and first question your thoughts and think that are you giving the correct answer or you are just hallucinating. Always try to give the most accurate answer. 
    The context is: <context> {context} </context>
    The user's question is: {question}. 
  """
)

document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', chain_type_kwargs={"prompt": prompt}, return_source_documents=True, retriever=retriever)

In [41]:
# Invoking the chain
response = retrieval_chain.invoke({'query': 'What is POVERTY IN METROPOLITAN AREAS?'})
response['result']

'According to the context, "POVERTY IN METROPOLITAN AREAS" refers to the information provided in Figure 4, which shows the percentage of people in poverty in 2021 and 2022 for the 25 most populous metropolitan areas. Additionally, Appendix Table 2 provides the estimated number and percentage of people in poverty in 2021 and 2022 for these areas. The context also mentions that some metropolitan areas, such as Washington, DC, and Denver, had among the lowest poverty rates, while others, such as Houston, San Antonio, and Detroit, had higher poverty rates. However, the context does not provide a specific definition of "POVERTY IN METROPOLITAN AREAS" beyond this information.'